# Notebook 0: Download Sample Data

This notebook downloads the raw imaging data used in the `liv_zones` tutorial
directly from figshare, so you can run the rest of the pipeline on real images.

**Dataset:** [Raw images for *Spatial single-cell Organellomics*](https://janelia.figshare.com/articles/dataset/Raw_images_for_Spatial_single-cell_Organellomics_reveals_nutrient_dependent_hepatocyte_heterogeneity_and_predicts_pathophysiological_status_i_i_i_in_vivo_i_/31863250)
(figshare article `31863250`).

The raw data is one TIF per channel per z-slice, named:
```
Region 1_Merged--Z{zz}--C{cc}.tif
```

**Channel mapping:**

| File channel | Structure |
|:---:|:---|
| C00 | Mitochondria |
| C01 | Actin |
| C02 | Lipid droplets |
| C03 | Peroxisomes |

> **Heads up — file sizes.** Each channel TIF is ~385 MB, so a single z-slice
> (4 channels) is roughly **1.5 GB**. The full lobule (51 z-slices) is ~88 GB.
> By default this notebook downloads **only one z-slice** — enough to run
> Notebook 1. Adjust `Z_SLICES` below if you want more.

Run this notebook first, then open Notebook 1, which loads the four channels for
your chosen z-slice.

## 1. Query the dataset

We use the public figshare API (no account required) to list the files in the
article and discover which z-slices and channels are available. This only reads
metadata — nothing is downloaded yet.

In [ ]:
import re
import json
import urllib.request
from pathlib import Path

# ── Configuration ────────────────────────────────────────────────
ARTICLE_ID = 31863250                       # figshare article to download from
DATA_DIR   = Path('figshare_data')          # where downloaded TIFs are saved
DATA_DIR.mkdir(parents=True, exist_ok=True)

# The main liver-lobule series; files look like 'Region 1_Merged--Z00--C00.tif'.
NAME_PATTERN = re.compile(r'--Z(\d+)--C(\d+)\.tif$', re.IGNORECASE)

# ── Fetch article metadata ────────────────────────────────────
api_url = f'https://api.figshare.com/v2/articles/{ARTICLE_ID}'
with urllib.request.urlopen(api_url, timeout=60) as resp:
    article = json.load(resp)

print(article['title'])
print(f"{len(article['files'])} files in article {ARTICLE_ID}\n")

# ── Index the per-channel files by (z-slice, channel) ───────────────────
# files_by_z[z][c] -> file metadata dict (name, size, download_url, ...)
files_by_z = {}
for f in article['files']:
    m = NAME_PATTERN.search(f['name'])
    if m:
        z, c = int(m.group(1)), int(m.group(2))
        files_by_z.setdefault(z, {})[c] = f

z_slices = sorted(files_by_z)
channels = sorted({c for chans in files_by_z.values() for c in chans})
print(f'Z-slices available : {z_slices[0]}–{z_slices[-1]}  ({len(z_slices)} total)')
print(f'Channels available : {channels}')

## 2. Choose a z-slice and download

Pick which z-slice(s) to download. By default we grab a single slice from the
middle of the stack. To download more, list several values in `Z_SLICES`
(e.g. `[20, 25, 30]`); to download the whole stack, set `Z_SLICES = z_slices`
— but mind the ~88 GB total.

Files that are already present (and the right size) are skipped, so it is safe
to re-run this cell.

In [ ]:
# ── Which z-slice(s) to download ─────────────────────────────────
Z_SLICE  = z_slices[len(z_slices) // 2]   # default: middle slice
Z_SLICES = [Z_SLICE]                      # add more slices here if desired


def download_file(meta, dest, chunk=1 << 20):
    """Stream a figshare file to `dest`, skipping if already complete."""
    if dest.exists() and dest.stat().st_size == meta['size']:
        print(f'  skip  {dest.name} (already downloaded)')
        return
    size_mb = meta['size'] / 1e6
    print(f'  get   {dest.name} ({size_mb:.0f} MB)', end='', flush=True)
    tmp = dest.with_suffix(dest.suffix + '.part')
    got = 0
    with urllib.request.urlopen(meta['download_url'], timeout=120) as r, open(tmp, 'wb') as out:
        while True:
            block = r.read(chunk)
            if not block:
                break
            out.write(block)
            got += len(block)
            print(f'\r  get   {dest.name} ({size_mb:.0f} MB)  {got / meta["size"]:5.0%}', end='', flush=True)
    tmp.rename(dest)
    print('  done')


# ── Download the selected slice(s), all four channels each ───────────────
for z in Z_SLICES:
    if z not in files_by_z:
        print(f'Z{z:02d}: not available, skipping')
        continue
    print(f'Z{z:02d}:')
    for c in channels:
        meta = files_by_z[z][c]
        download_file(meta, DATA_DIR / meta['name'])

print(f'\nDone. Files saved to: {DATA_DIR.resolve()}')

## Summary

You now have the raw channel TIFs for your chosen z-slice in `DATA_DIR`.

Carry these two values over to **Notebook 1** (`01_preprocessing.ipynb`):

```python
DATA_DIR = Path('figshare_data')   # the folder used above
Z_SLICE  = ...                     # the slice you downloaded
```

The cell below prints them for convenience.

In [ ]:
print(f"DATA_DIR = Path('{DATA_DIR}')")
print(f'Z_SLICE  = {Z_SLICE}')